In [ ]:
import Pkg
Pkg.activate(joinpath(pwd(), "."))
Pkg.instantiate()


In [ ]:
using Random, CSV, DataFrames, HDF5, JLD2, Statistics, StatsBase, Normalization
using CairoMakie
using Images: imresize
using ImageFiltering: imfilter, KernelFactors


In [ ]:
# Preprocessing settings (hard-coded for now)
target_height = 64
target_width = 64
zscore_timepoints = true
resize_antialias = true
low_pass_factor = 0.75

println("Settings (hard-coded):")
println("  target_height=", target_height, ", target_width=", target_width)
println("  zscore_timepoints=", zscore_timepoints)
println("  resize_antialias=", resize_antialias, ", low_pass_factor=", low_pass_factor)


In [ ]:
# Epoching parameters (hard-coded for this dataset)
# Assumes running from notebooks/model_test

data_dir = pwd()

sampling_rate = 512  # Hz
pre_stim_s = 0.5     # seconds
pre_samples = Int(round(pre_stim_s * sampling_rate))


In [ ]:
# Load real-data events + ERP single-trial data (hard-coded paths/keys)
# Assumes running from notebooks/model_test

fixations_data_dir = joinpath(data_dir, "real_data_sets", "fixations_dataset")
events = CSV.read(joinpath(fixations_data_dir, "events.csv"), DataFrame)

fid = h5open(joinpath(fixations_data_dir, "data_fixations.hdf5"), "r")
erps_fix = read(fid["data"]["data_fixations.hdf5"])
close(fid)

n_time = size(erps_fix, 2)
post_samples = n_time - pre_samples - 1

time_axis = collect((-pre_samples:post_samples) ./ sampling_rate)
epoch_window = (time_axis[1], time_axis[end])
time_zero_idx = pre_samples + 1

println("events rows: ", nrow(events))
println("erps_fix size (channels x time x trials): ", size(erps_fix))
println("sampling_rate: ", sampling_rate, " Hz")
println("epoch_window (s): ", epoch_window)


In [ ]:
# Helpers: sorting + z-score per timepoint + low-pass filter + resize
const FILTER_BORDER = "reflect"
const NON_SORT_COLS = Set([:id, :picID, :trialnum, :stim_set, :stim_file])

function sortvalues_from(df::DataFrame, col::Symbol)
    v = df[!, col]
    if eltype(v) <: Number
        return Float64.(v)
    end
    return collect(v)
end

function select_sort_cols(df::DataFrame; exclude = NON_SORT_COLS)
    cols = Symbol[]
    for c in Symbol.(names(df))
        c in exclude && continue
        v = df[!, c]
        length(unique(v)) < 2 && continue
        push!(cols, c)
    end
    return cols
end

function zscore_timepoints_mat(data_time_trials)
    z = Normalization.normalize(Float64.(data_time_trials), ZScore; dims = 2)
    return Float32.(z)
end

function lowpass_resize(img_trials_time; target_height, target_width, resize_antialias, low_pass_factor)
    out = Float32.(img_trials_time)
    if resize_antialias && low_pass_factor > 0
        sigma = (low_pass_factor * size(out, 1) / target_height,
                 low_pass_factor * size(out, 2) / target_width)
        out = Float32.(imfilter(out, KernelFactors.gaussian(sigma), FILTER_BORDER))
    end
    return Float32.(imresize(out, (target_height, target_width)))
end

function erp_image_from_channel(erps, events::DataFrame, channel::Int, sort_col::Symbol;
        target_height, target_width, resize_antialias, low_pass_factor, zscore_timepoints::Bool, time_zero_idx::Int)
    data = Float32.(erps[channel, time_zero_idx:end, :])  # time x trials (t >= 0)
    sortvals = sortvalues_from(events, sort_col)
    n_trials = size(data, 2)
    if length(sortvals) != n_trials
        @warn "Trial count mismatch (events vs ERP trials)" sort_col=sort_col event_rows=length(sortvals) erp_trials=n_trials
        error("Mismatch between events and ERP trials; aborting.")
    end
    order = sortperm(sortvals)
    data_sorted = data[:, order]
    if zscore_timepoints
        data_sorted = zscore_timepoints_mat(data_sorted)
    else
        data_sorted = Float32.(data_sorted)
    end
    img_trials_time = permutedims(data_sorted, (2, 1))
    img_resized = lowpass_resize(img_trials_time;
        target_height = target_height,
        target_width = target_width,
        resize_antialias = resize_antialias,
        low_pass_factor = low_pass_factor,
    )
    return img_resized
end


In [ ]:
# Overview: per sort-variable 20 random ERP images (each with its own color scale)
Random.seed!(time_ns())
rng = Random.default_rng()

sort_cols = select_sort_cols(events)
println("Sortable columns (", length(sort_cols), "):")
println(sort_cols)

if isempty(sort_cols)
    @warn "No sortable columns found in events"
else
    n_channels = size(erps_fix, 1)
    n_trials = size(erps_fix, 3)
    n_per_sort = 20
    ncols = 5

    x_vals = 1:target_width
    y_vals = 1:target_height
    x_ticks = ([1, target_width], ["1", string(target_width)])
    y_ticks = ([1, target_height], ["1", string(target_height)])

    erp_cmap = :balance

    for sort_col in sort_cols
        n_pick = min(n_per_sort, n_channels)
        picks = randperm(rng, n_channels)[1:n_pick]
        nrows = cld(n_pick, ncols)
        fig = Figure(size = (260 * ncols, 240 * nrows + 40), figure_padding = (20, 10, 30, 10))
        rowgap!(fig.layout, 12)
        colgap!(fig.layout, 12)
        Label(fig[0, :], "sort=" * string(sort_col) * " | 20 random channels"; tellwidth = false, fontsize = 14)

        for (i, ch) in enumerate(picks)
            row = cld(i, ncols)
            col = i - (row - 1) * ncols
            img = erp_image_from_channel(erps_fix, events, ch, sort_col;
                target_height = target_height,
                target_width = target_width,
                resize_antialias = resize_antialias,
                low_pass_factor = low_pass_factor,
                zscore_timepoints = zscore_timepoints,
                time_zero_idx = time_zero_idx,
            )
            vmax = max(quantile(vec(abs.(img)), 0.99), eps(Float32))
            cr = (-vmax, vmax)
            ax = Axis(fig[row, col];
                title = string(sort_col) * " | ch " * string(ch),
                titlesize = 10,
                xticks = x_ticks,
                yticks = y_ticks,
                xticklabelsize = 8,
                yticklabelsize = 8,
                aspect = AxisAspect(1),
            )
            heatmap!(ax, x_vals, y_vals, permutedims(img, (2, 1)); colormap = erp_cmap, colorrange = cr)
        end
        display(fig)
    end
end


In [ ]:
# # Export ERP images for Label Studio (channel + sort variable metadata)
# # Default: export 100 images total (distributed across sort variables).

# n_export_total = 100

# # Export into the same folder as this notebook
# notebook_dir = isfile(joinpath(pwd(), "data_vis.ipynb")) ? pwd() : joinpath(pwd(), "notebooks", "model_test")
# export_root = joinpath(notebook_dir, "erp_labelstudio_exports")
# mkpath(export_root)

# sort_cols = select_sort_cols(events)

# n_sort = length(sort_cols)
# n_channels = size(erps_fix, 1)

# base = fld(n_export_total, n_sort)
# remainder = n_export_total - base * n_sort

# x_vals = 1:target_width
# y_vals = 1:target_height
# x_ticks = ([1, target_width], ["1", string(target_width)])
# y_ticks = ([1, target_height], ["1", string(target_height)])
# erp_cmap = :balance

# manifest = DataFrame(image = String[], channel = Int[], sort_col = String[])

# for (i, sort_col) in enumerate(sort_cols)
#     n_pick = base + (i <= remainder ? 1 : 0)
#     n_pick = min(n_pick, n_channels)

#     picks = randperm(n_channels)[1:n_pick]
#     sort_slug = replace(string(sort_col), r"[^A-Za-z0-9_-]" => "_")

#     for ch in picks
#         img = erp_image_from_channel(erps_fix, events, ch, sort_col;
#             target_height = target_height,
#             target_width = target_width,
#             resize_antialias = resize_antialias,
#             low_pass_factor = low_pass_factor,
#             zscore_timepoints = zscore_timepoints,
#             time_zero_idx = time_zero_idx,
#         )

#         vmax = max(quantile(vec(abs.(img)), 0.99), eps(Float32))
#         cr = (-vmax, vmax)

#         fig = Figure(size = (260, 240), figure_padding = (20, 10, 30, 10))
#         ax = Axis(fig[1, 1];
#             title = string(sort_col) * " | ch " * string(ch),
#             titlesize = 10,
#             xticks = x_ticks,
#             yticks = y_ticks,
#             xticklabelsize = 8,
#             yticklabelsize = 8,
#             aspect = AxisAspect(1),
#         )
#         heatmap!(ax, x_vals, y_vals, permutedims(img, (2, 1)); colormap = erp_cmap, colorrange = cr)

#         filename = "ch" * string(ch) * "__sort-" * sort_slug * ".png"
#         out_path = joinpath(export_root, filename)
#         CairoMakie.save(out_path, fig)

#         push!(manifest, (image = out_path, channel = ch, sort_col = string(sort_col)))
#     end
# end

# manifest_path = joinpath(export_root, "labelstudio_manifest.csv")
# CSV.write(manifest_path, manifest)
# println("Exported ", nrow(manifest), " images to ", export_root)
# println("Manifest: ", manifest_path)
